# Randomized value LULC elements for pre-mapathon modular reference data

The values of each elements are properties are randomized based on a range that is set for each of the LULC classes

In [ ]:
# import ee

# ee.Authenticate() 
# ee.Initialize()

## Set seed for constant randomization

In [ ]:
import random
import numpy as np

RANDOM_SEED      = 999

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

# Upload class-labelled points

In [ ]:
# Upload class-labelled reference data
import geopandas as gpd

INPUT_SHAPEFILE  = "../data/modular_mapping_approach/pagaralam_test/Pagar_Alam_Ext_Points.shp"

gdf = gpd.read_file(INPUT_SHAPEFILE)

gdf.head()

# Upload CSV of the randomization ruleset per class

In [ ]:
import pandas as pd

RULES_CSV = "../data/modular_mapping_approach/attribute_random_rules_kalimantansumatra.csv"
CLASS_FIELD = "LULC_24"          # field in the shapefile holding the class name

rules_df = pd.read_csv(RULES_CSV, dtype=str, keep_default_na=False)
rules_df = rules_df.set_index(CLASS_FIELD)

# Build {class_name: {column_name: rule_string}} from the CSV, dropping empty cells
ATTRIBUTE_RULES = {
    class_name: {col: val for col, val in row.items() if str(val).strip() != ""}
    for class_name, row in rules_df.to_dict(orient="index").items()
}

# Sanity check: list all attribute columns that will be created
ATTRIBUTE_COLUMNS = sorted({col for rules in ATTRIBUTE_RULES.values() for col in rules})
print(f"{len(ATTRIBUTE_RULES)} classes loaded from '{RULES_CSV}', "
      f"{len(ATTRIBUTE_COLUMNS)} attribute columns will be added:")
print(ATTRIBUTE_COLUMNS)

# Check shapefile and csv class fields

## Sampling helper functions

In [ ]:
import re
import random

_RANGE_RE = re.compile(r"^\s*(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)\s*$")


def sample_rule(rule):
    """Draw one value from a single rule string.

    - "low-high"           -> random float uniformly in [low, high], rounded to 1 decimal
    - "optionA/optionB/..." -> random.choice of the options
    - "fixedValue"          -> returned as-is
    """
    rule = str(rule).strip()

    m = _RANGE_RE.match(rule)
    if m:
        low, high = float(m.group(1)), float(m.group(2))
        if low > high:
            low, high = high, low
        return round(random.uniform(low, high), 1)

    if "/" in rule:
        options = [o.strip() for o in rule.split("/") if o.strip()]
        return random.choice(options)

    return rule


def sample_class_attributes(class_name, rules_table=ATTRIBUTE_RULES):
    """Return a dict of {column: sampled_value} for a given class_name.

    Unrecognized classes return all-NaN/None values (and print a one-time warning
    via the caller) so the row is preserved but flagged for manual review.
    """
    if class_name not in rules_table:
        return {col: None for col in ATTRIBUTE_COLUMNS}
    return {col: sample_rule(rule) for col, rule in rules_table[class_name].items()}


In [ ]:
assert CLASS_FIELD in gdf.columns, (
    f"'{CLASS_FIELD}' not found in shapefile columns: {list(gdf.columns)}. "
    "Update CLASS_FIELD in the CONFIG cell to match your field name."
)

unmatched = sorted(set(gdf[CLASS_FIELD].unique()) - set(ATTRIBUTE_RULES.keys()))
if unmatched:
    print("WARNING: the following class_name values in the shapefile have no matching "
          "rule and will get empty attribute columns:")
    for c in unmatched:
        print(f"  - {c!r}")
else:
    print("All class_name values in the shapefile have matching rules.")


# Generate random values to the columns per points

In [ ]:
sampled_rows = gdf[CLASS_FIELD].apply(sample_class_attributes)
attr_df = pd.DataFrame(list(sampled_rows), index=gdf.index)

# Merge sampled attributes into the original GeoDataFrame (geometry stays untouched)
gdf_out = gdf.join(attr_df)
gdf_out.head()

# Save the output to local

In [ ]:
from pathlib import Path

OUTPUT_SHAPEFILE = "../data/temp/pagaralam_test_revengineer.shp"
OUTPUT_CSV = "../data/temp/pagaralam_test_revengineer.csv"

Path(OUTPUT_SHAPEFILE).parent.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
gdf_out.to_file(OUTPUT_SHAPEFILE)  # Note: .shp truncates field names to 10 chars (see Notes below)
print(f"Saved {len(gdf_out)} points with {len(ATTRIBUTE_COLUMNS)} new attribute columns to:")
print(OUTPUT_SHAPEFILE)

# Save the output to CSV (geometry column is dropped since CSV can't store geometries)
gdf_out.to_csv(OUTPUT_CSV, index=False)